# DeepStego — experimentos en Kaggle

Activa **GPU** e **Internet**. Adjunta DIV2K y el dataset privado creado con el **nuevo** `outputs/deepstego-kaggle.zip`. El ZIP incluye el código actualizado y el `best.pt` original (época 3). Cambiar solo este notebook no actualiza el código del dataset.

El entrenamiento original colapsó desde la época 4: el QR se volvió blanco. Este notebook parte de `best.pt`, reduce la tasa y mide lectura exacta y SSIM cada época. **No uses el `last.pt` original para el ajuste fino.**

Ejecuta las celdas en orden: referencia → un experimento → comparación. Estas métricas son de validación; COCO se reserva para la evaluación final.

In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path
from zipfile import ZipFile

if any(name == "deepstego" or name.startswith("deepstego.") for name in sys.modules):
    raise RuntimeError("Reinicia el kernel antes de cargar otra vez el código.")

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
WORK_ROOT.mkdir(parents=True, exist_ok=True)

projects = [
    path.parent
    for path in INPUT_ROOT.rglob("pyproject.toml")
    if (path.parent / "src/deepstego/train.py").is_file()
]
if not projects:
    archives = [
        path
        for path in INPUT_ROOT.rglob("*")
        if path.is_file() and path.name.casefold() == "deepstego-kaggle.zip"
    ]
    if not archives:
        raise RuntimeError(
            "Falta el código: adjunta deepstego-kaggle.zip como otro dataset, "
            "además de DeepStego-DIV2K. El notebook solo no contiene las redes."
        )
    if len(archives) > 1:
        raise RuntimeError(f"Hay varios ZIP de código; deja uno adjunto: {archives}")
    with ZipFile(archives[0]) as archive:
        archive.extractall(WORK_ROOT / "source-upload")
    projects = [
        path.parent
        for path in (WORK_ROOT / "source-upload").rglob("pyproject.toml")
        if (path.parent / "src/deepstego/train.py").is_file()
    ]
if len(projects) != 1:
    raise RuntimeError(f"Se esperaba una única copia del código: {projects}")

PROJECT_DIR = WORK_ROOT / "deepstego-code"
shutil.copytree(projects[0], PROJECT_DIR, dirs_exist_ok=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "qrcode[pil]", "scikit-image", "matplotlib"],
    check=True,
)
sys.path.insert(0, str(PROJECT_DIR / "src"))

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Activa un acelerador GPU en la configuración del notebook.")
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
import deepstego.train as training_module

if "init_from" not in training_module.TrainConfig.__dataclass_fields__:
    raise RuntimeError("Código antiguo: adjunta el ZIP actualizado y reinicia el kernel.")
print("Código cargado:", training_module.__file__)

## Datos fijos para comparar

Se esperan 800 fotografías de entrenamiento y 100 de validación. Los recortes y QR de entrenamiento cambian; la validación conserva los mismos pares con semilla 43, también en las evaluaciones PNG.

In [ ]:
from deepstego.data import DIV2KDataset, find_div2k_split

TRAIN_DIR = find_div2k_split(INPUT_ROOT, "DIV2K_train_HR")
VAL_DIR = find_div2k_split(INPUT_ROOT, "DIV2K_valid_HR")
SEED = 42
train_data = DIV2KDataset(TRAIN_DIR, training=True, seed=SEED)
val_data = DIV2KDataset(VAL_DIR, training=False, seed=SEED + 1)
print("Entrenamiento:", len(train_data), TRAIN_DIR)
print("Validación:", len(val_data), VAL_DIR)
if len(train_data) != 800 or len(val_data) != 100:
    raise RuntimeError(
        "Se esperaba DIV2K completo: 800 imágenes de entrenamiento y 100 de validación."
    )

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for index in range(4):
    sample = train_data[index % len(train_data)]
    axes[0, index].imshow(sample["cover"].permute(1, 2, 0))
    axes[1, index].imshow(sample["secret"][0], cmap="gray", vmin=0, vmax=1)
    axes[1, index].set_title(sample["text"], fontsize=8)
for axis in axes.flat:
    axis.axis("off")
plt.tight_layout()
plt.show()

## Elegir un experimento

El valor inicial `mse_lr1e4_beta1` entrena **10 épocas nuevas** desde los pesos originales, con Adam nuevo, LR `1e-4`, lote 16 y β=1. El historial comienza en 1; los recortes continúan desde la época del checkpoint inicial.

Cambia `EXPERIMENT` para comparar β desde el mismo punto de partida. `bce_lr1e4_beta01` prueba BCE sobre logits: β=0.1 es exploratorio, no un valor validado. Las pérdidas totales MSE/BCE no son comparables; compara lectura, PSNR y SSIM.

**Reanudar una corrida nueva:** indica su `last.pt` en `RESUME` y aumenta `EPOCHS` al total deseado de esa corrida. Mantén sus hiperparámetros y los archivos `best.pt` y, si existe, `best_quality.pt` junto a `last.pt`. Se restauran optimizador y scheduler.

Para cambiar hiperparámetros inicia otro experimento sin `RESUME`. Una carpeta con checkpoints no se sobrescribe. Para repetir una configuración desde cero cambia `RUN_NAME`.

In [ ]:
from deepstego.train import TrainConfig, train

EXPERIMENTS = {
    "mse_lr1e4_beta1": {"secret_loss": "mse", "beta": 1.0},
    "mse_lr1e4_beta05": {"secret_loss": "mse", "beta": 0.5},
    "mse_lr1e4_beta02": {"secret_loss": "mse", "beta": 0.2},
    "bce_lr1e4_beta01": {"secret_loss": "bce", "beta": 0.1},
}
EXPERIMENT = "mse_lr1e4_beta1"
RUN_NAME = EXPERIMENT
EPOCHS = 10
RESUME = None
INITIAL_CHECKPOINT = PROJECT_DIR / "checkpoints" / "best.pt"
OUTPUT_DIR = WORK_ROOT / "experiments" / RUN_NAME

if not INITIAL_CHECKPOINT.is_file():
    raise FileNotFoundError("Falta best.pt: vuelve a crear y adjuntar el ZIP actualizado.")
initial = torch.load(INITIAL_CHECKPOINT, map_location="cpu", weights_only=True)
print("Referencia:", INITIAL_CHECKPOINT, "| época:", initial["epoch"])
config = TrainConfig(
    train_dir=str(TRAIN_DIR),
    val_dir=str(VAL_DIR),
    output_dir=str(OUTPUT_DIR),
    epochs=EPOCHS,
    batch_size=16,
    learning_rate=1e-4,
    width=initial["config"]["width"],
    crops_per_image=4,
    num_workers=2,
    seed=SEED,
    device="cuda",
    resume=RESUME,
    init_from=str(INITIAL_CHECKPOINT) if RESUME is None else None,
    lr_patience=2,
    early_stop_patience=5,
    **EXPERIMENTS[EXPERIMENT],
)
print(config)

## Evaluar la referencia

Se evalúan los 100 pares guardando y recargando PNG. Cada ejecución crea una carpeta nueva. Conserva esta referencia para comprobar si el ajuste mejora realmente al modelo original.

In [ ]:
from tempfile import mkdtemp

from deepstego.evaluate import evaluate

BASELINE_DIR = Path(mkdtemp(prefix="baseline-", dir=WORK_ROOT))
baseline = evaluate(
    INITIAL_CHECKPOINT,
    VAL_DIR,
    BASELINE_DIR,
    count=len(val_data),
    seed=SEED + 1,
    device="cuda",
)
print(baseline)
print("Referencia guardada en:", BASELINE_DIR)

## Entrenar y vigilar la recuperación

Cada época mide PSNR, SSIM y lectura exacta en toda la validación. Aquí se utiliza cuantización a 8 bits equivalente a PNG; la evaluación posterior sí escribe y recarga archivos.

El scheduler reduce la tasa a la mitad cuando se estanca la pérdida (`lr_patience=2`). Se detiene tras 5 épocas sin mejorar o si al menos el 95 % de los QR quedan totalmente blancos/negros saturados y ninguno se lee.

- `best.pt`: menor pérdida de validación de esta corrida.
- `best_quality.pt`: mayor PSNR entre épocas con lectura ≥90 % y SSIM ≥0,90. Solo existe si se alcanzan esas condiciones; **hay que comprobar además PSNR ≥30 dB**.
- `last.pt`: última época, para reanudar una corrida que no haya colapsado.

El checkpoint original permanece separado. Si el ajuste empeora, conserva esa referencia.

In [ ]:
history = train(config)
last = torch.load(OUTPUT_DIR / "last.pt", map_location="cpu", weights_only=True)
print("Motivo de parada:", last.get("stop_reason") or "Épocas solicitadas completadas.")
print("Resultados:", OUTPUT_DIR)

In [ ]:
epochs = [row["epoch"] for row in history]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for split in ("train", "val"):
    axes[0, 0].plot(epochs, [row[split]["loss"] for row in history], label=split)
    axes[0, 1].plot(epochs, [row[split]["secret_mse"] for row in history], label=split)
    axes[0, 2].plot(epochs, [row[split]["psnr"] for row in history], label=split)
axes[1, 0].plot(epochs, [row["val"].get("ssim", float("nan")) for row in history], label="val")
axes[1, 1].plot(
    epochs, [100 * row["val"].get("reading_rate", float("nan")) for row in history], label="val"
)
axes[1, 2].plot(epochs, [row.get("learning_rate", float("nan")) for row in history], label="LR")
axes[0, 2].axhline(30, color="gray", linestyle="--")
axes[1, 0].axhline(0.90, color="gray", linestyle="--")
axes[1, 1].axhline(90, color="gray", linestyle="--")
axes[1, 1].set_ylim(0, 105)
axes[1, 2].set_yscale("log")
titles = ("Pérdida total", "MSE del QR", "PSNR (dB)", "SSIM", "Lectura exacta (%)", "Learning rate")
for axis, title in zip(axes.flat, titles):
    axis.set_title(title)
    axis.set_xlabel("Época del experimento")
    axis.legend()
    axis.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "curves.png")
plt.show()

## Comparar con PNG reales

Evalúa `best.pt` y, si existe, `best_quality.pt` sobre los mismos 100 pares de la referencia. Guarda CSV por imagen, resúmenes y ejemplos de éxito/fallo. Esto sigue siendo **validación**; COCO se utiliza al terminar de elegir el modelo.

In [ ]:
import csv

evaluation_root = Path(mkdtemp(prefix="png-validation-", dir=OUTPUT_DIR))
summaries = {"original": baseline}
for name in ("best.pt", "best_quality.pt"):
    candidate = OUTPUT_DIR / name
    if candidate.is_file():
        summaries[name] = evaluate(
            candidate,
            VAL_DIR,
            evaluation_root / candidate.stem,
            count=len(val_data),
            seed=SEED + 1,
            device="cuda",
        )
rows = []
for name, summary in summaries.items():
    rows.append(
        {
            "model": name,
            "epoch": summary["epoch"],
            "psnr": summary["psnr_mean"],
            "ssim": summary["ssim_mean"],
            "reading_rate": summary["reading_rate"],
            "meets_all": all(summary["criteria"].values()),
        }
    )
    print(
        f"{name}: PSNR={summary['psnr_mean']:.2f} dB | "
        f"SSIM={summary['ssim_mean']:.3f} | lectura={summary['reading_rate']:.1%} | "
        f"cumple todas las metas={all(summary['criteria'].values())}"
    )
with (evaluation_root / "comparison.csv").open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
print("Comparación guardada:", evaluation_root)

In [ ]:
import numpy as np
from PIL import Image

from deepstego.models import HideNet, RevealNet, quantize_8bit
from deepstego.qr import read_qr

device = torch.device(config.device)
SELECTED_CHECKPOINT = OUTPUT_DIR / "best.pt"
checkpoint = torch.load(SELECTED_CHECKPOINT, map_location=device, weights_only=True)
hide = HideNet(checkpoint["config"]["width"]).to(device)
reveal = RevealNet(checkpoint["config"]["width"]).to(device)
hide.load_state_dict(checkpoint["hide"])
reveal.load_state_dict(checkpoint["reveal"])
hide.eval()
reveal.eval()

sample = val_data[0]
cover = sample["cover"].unsqueeze(0).to(device)
secret = sample["secret"].unsqueeze(0).to(device)
with torch.inference_mode():
    container = quantize_8bit(hide(cover, secret))
    pixels = container[0].permute(1, 2, 0).mul(255).round().byte().cpu().numpy()
    Image.fromarray(pixels).save(OUTPUT_DIR / "container.png")
    with Image.open(OUTPUT_DIR / "container.png") as png:
        loaded = torch.from_numpy(np.array(png.convert("RGB"))).permute(2, 0, 1).float() / 255
    recovered = reveal(loaded.unsqueeze(0).to(device))[0, 0].cpu()

result = read_qr(recovered)
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
images = [sample["cover"].permute(1, 2, 0), sample["secret"][0], pixels, recovered]
for axis, image, title in zip(axes, images, ("Portadora", "QR original", "PNG", "QR revelado")):
    axis.imshow(image, cmap="gray", vmin=0, vmax=1)
    axis.set_title(title)
    axis.axis("off")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "examples.png")
plt.show()
print("Texto esperado:", sample["text"])
print("Texto recuperado:", result.text)
print("Coincide:", result.text == sample["text"])
print("Época seleccionada:", checkpoint["epoch"])
print("Outputs:", OUTPUT_DIR)

## Conservar y continuar

Descarga todos los outputs. Los checkpoints guardan configuración, historial, optimizador y scheduler. También se conservan `history.json`, curvas y evaluaciones PNG. Guarda una versión ejecutada del notebook para conservar sus salidas.

La celda siguiente empaqueta el experimento y su referencia. Para otro β cambia `EXPERIMENT` y `RUN_NAME`, deja `RESUME=None` y parte de los pesos originales.

**Save & Run All** vuelve a ejecutar todo en otra sesión: revisa `RESUME` y la salida. Al actualizar el ZIP reinicia el kernel.

Una vez elegido el modelo con DIV2K de validación, utiliza `deepstego.evaluate` sobre 1.000 imágenes COCO y semilla 44 para la evaluación final. No ajustes hiperparámetros con esos resultados.

In [ ]:
from IPython.display import FileLink, display

archive_path = WORK_ROOT / f"{RUN_NAME}-results.zip"
with ZipFile(archive_path, "w") as archive:
    for folder in (OUTPUT_DIR, BASELINE_DIR):
        for path in sorted(folder.rglob("*")):
            if path.is_file():
                archive.write(path, Path(folder.name) / path.relative_to(folder))
display(FileLink(str(archive_path)))